# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook scores every page with the **validated model from `w05_model.ipynb`** -- Logistic
Regression, the pick `w06_validation_audit.ipynb` confirmed wins at the K values a content manager
actually works through, evaluated honestly with a client-grouped holdout (never a random row
split). Two models exist here on purpose:

1. A **holdout copy** (trained on 80% of clients, tested on the other 20%, zero client overlap) --
   used only to print the honest, already-audited expected performance below as a receipt.
2. The **final scoring model** (trained on all 30,000 rows) -- used to score every page and build
   the ranked queue. This is normal practice (use all available data for the deployed model,
   disclose performance from the held-out design), not a second, less-honest number.

**Reason codes and the archetype -> action map** (extends the two reason codes from
`w04_baseline_score.ipynb`'s rule with the model's risk score as a third axis):

| Archetype | Condition | Reason code | Action |
|---|---|---|---|
| Declining & Visible -- Click Laggard | high model risk + visible + good position + CTR below its own tier's mean | `ctr_fix_opportunity` | `optimize_title_meta` |
| Declining & Visible -- Needs Refresh | high model risk + visible, doesn't clear the CTR-laggard bar | `stale_needs_refresh` | `refresh_content` |
| Declining but Quiet | high model risk + low visibility | `low_visibility_monitor` | `monitor_only` |
| Stable | everything else | `not_flagged` | `no_action` |

"High model risk" is the **top quartile** of predicted decline probability -- a ranking cut, not a
claim that 0.65 is some universal danger threshold. Priority within the flagged rows is
`decline_proba x value_at_stake`, where `value_at_stake = clicks_90d x cpc` -- the same
defensible click-equivalent-value proxy the FlyRank paper uses (clicks x CPC, never
impressions x CPC, since impressions aren't billable).

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

# Same leakage-safe feature set as w05_model.ipynb / w06_validation_audit.ipynb
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]

num_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
enc_frame = pd.get_dummies(cat_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num_frame.reset_index(drop=True), enc_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
groups = df["client_id"]

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    k = min(k, len(y_true))
    return y_true[order[:k]].mean()

# --- Receipt: honest, client-grouped holdout performance (same design as w06) ---
gs = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gs.split(X, y, groups))
holdout_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
holdout_pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
holdout_proba = holdout_pipe.predict_proba(X.iloc[test_idx])[:, 1]
y_test = y.iloc[test_idx]

expected_performance = {"test_base_rate": round(y_test.mean(), 3)}
for k in [20, 50, 100]:
    expected_performance[f"precision_at_{k}"] = round(precision_at_k(y_test, holdout_proba, k), 3)
expected_performance["roc_auc"] = round(roc_auc_score(y_test, holdout_proba), 3)
expected_performance["avg_precision"] = round(average_precision_score(y_test, holdout_proba), 3)

print("Expected out-of-sample performance (client-grouped holdout, matches w06's honest split):")
for k, v in expected_performance.items():
    print(f"  {k}: {v}")

# --- Final scoring model: trained on ALL data, used to score every page in the queue ---
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
final_pipe.fit(X, y)
df["decline_proba"] = final_pipe.predict_proba(X)[:, 1]
print(f"\nScored {len(df):,} pages. decline_proba range: {df['decline_proba'].min():.3f} - {df['decline_proba'].max():.3f}")

Expected out-of-sample performance (client-grouped holdout, matches w06's honest split):
  test_base_rate: 0.511
  precision_at_20: 0.65
  precision_at_50: 0.72
  precision_at_100: 0.66
  roc_auc: 0.583
  avg_precision: 0.577



Scored 30,000 pages. decline_proba range: 0.005 - 1.000


In [2]:
# --- Archetype / reason code / action mapping ---
risk_threshold = df["decline_proba"].quantile(0.75)  # top quartile = "high risk" ranking cut
high_risk = df["decline_proba"] >= risk_threshold
visible = df["impression_tier"].isin(["moderate", "good", "excellent"])
good_position = df["position_tier"].isin(["top_3", "page_1", "striking"])
tier_mean_ctr = df.groupby("position_tier")["ctr"].transform("mean")
ctr_lags_tier = df["ctr"] < tier_mean_ctr

conditions = [
    high_risk & visible & good_position & ctr_lags_tier,
    high_risk & visible,
    high_risk & ~visible,
]
df["archetype"] = np.select(
    conditions,
    ["Declining & Visible -- Click Laggard", "Declining & Visible -- Needs Refresh", "Declining but Quiet"],
    default="Stable",
)
df["reason_code"] = np.select(
    conditions,
    ["ctr_fix_opportunity", "stale_needs_refresh", "low_visibility_monitor"],
    default="not_flagged",
)
df["action"] = np.select(
    conditions,
    ["optimize_title_meta", "refresh_content", "monitor_only"],
    default="no_action",
)

# Value proxy: clicks x CPC, the defensible captured-value formula (never impressions x CPC)
df["value_at_stake"] = (df["clicks_90d"] * df["cpc"]).round(2)
df["priority_score"] = (df["decline_proba"] * df["value_at_stake"]).round(2)

print(f"High-risk threshold (75th percentile of decline_proba): {risk_threshold:.3f}\n")
print("Archetype counts:")
print(df["archetype"].value_counts())
print("\nAction counts:")
print(df["action"].value_counts())

queue = (
    df[df["action"] != "no_action"]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)
display_cols = ["content_id", "client_id", "archetype", "reason_code", "action",
                 "decline_proba", "value_at_stake", "priority_score",
                 "impressions_90d", "avg_position", "ctr", "freshness_tier"]
pd.set_option("display.width", 200)
print(f"\nFlagged queue: {len(queue):,} of {len(df):,} pages ({len(queue)/len(df):.1%})")
queue[display_cols].head(10)

High-risk threshold (75th percentile of decline_proba): 0.650

Archetype counts:
archetype
Stable                                  22500
Declining & Visible -- Click Laggard     4045
Declining & Visible -- Needs Refresh     2092
Declining but Quiet                      1363
Name: count, dtype: int64

Action counts:
action
no_action              22500
optimize_title_meta     4045
refresh_content         2092
monitor_only            1363
Name: count, dtype: int64

Flagged queue: 7,500 of 30,000 pages (25.0%)


,content_id,client_id,archetype,reason_code,action,decline_proba,value_at_stake,priority_score,impressions_90d,avg_position,ctr,freshness_tier
0,content_a4087c89f66f,client_19581e27de,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.668540,348.91,233.26,7673,4.8,0.30,0-30
1,content_1d94287ddba7,client_3fdba35f04,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.703985,219.60,154.60,3419,7.2,0.53,91-180
2,content_4276cb52f7dd,client_3fdba35f04,Declining & Visible -- Needs Refresh,stale_needs_refresh,refresh_content,0.732995,207.87,152.37,4996,3.6,0.82,91-180
3,content_72e800a9c214,client_3fdba35f04,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.763087,177.12,135.16,13790,8.2,0.12,91-180
4,content_9cfb7d21dd56,client_f369cb89fc,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.695674,184.68,128.48,21346,5.5,0.13,0-30
5,content_fa895bac6d5f,client_7f2253d7e2,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.759351,164.45,124.88,3718,16.7,0.30,0-30
6,content_0390a273c940,client_a88a7902cb,Declining & Visible -- Needs Refresh,stale_needs_refresh,refresh_content,0.662877,174.60,115.74,10484,13.1,0.34,0-30
7,content_6c13999dcf48,client_3fdba35f04,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.751875,148.68,111.79,7319,13.9,0.29,0-30
8,content_ded62646b9e8,client_7f2253d7e2,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.764079,139.08,106.27,7288,5.5,0.16,0-30
9,content_199a6be9295f,client_b4944c6ff0,Declining & Visible -- Click Laggard,ctr_fix_opportunity,optimize_title_meta,0.663846,142.15,94.37,1035,9.2,0.48,0-30


## 2. Intended use and limits

**Intended use.** A weekly triage aid for FlyRank content managers on Lane 2 (refresh/content
opportunity scoring): rank pages so a human reviews the highest-priority ones first, with a
reason code that says *why* a page is flagged, not just *that* it is. It is a ranking and
prioritization tool -- not a scoring authority, and not a client-facing report on its own.

**Limits, stated plainly (the receipts are printed below, not just asserted):**

- **Modest, honestly-measured skill.** The client-grouped holdout ROC-AUC is ~0.58 -- real
  but modest lift over a coin flip, and precision@K only meaningfully beats the ~51-55% base
  rate at the K values a manager actually works through (per `w06`'s before/after comparison).
  This model earns a rank-ordering role, not an autonomous-decision role.
- **Cross-sectional, not causal.** Every score comes from one snapshot. `w06_validation_audit.ipynb`
  already flagged that "refresh" comparisons in the source research paper conflate selection with
  effect -- the same caution applies here: `decline_proba` is an observed pattern match, not proof
  that any specific action will reverse a page's trajectory.
- **Thin buckets exist.** `w04_baseline_score.ipynb`'s signal check found the `31-90` and `181+`
  freshness tiers sit at only ~175 rows each (vs 20,480 and 9,171 for the other two) -- any reason
  code leaning on those tiers is standing on a much smaller evidence base and should be flagged as
  such to the reviewer.
- **One snapshot in time, not validated across time.** The split here (and in `w05`/`w06`) is
  client-grouped, never time-based -- this dataset has no repeated per-page observations over
  time, so seasonal drift or the model's shelf life going forward is untested, not just unmeasured.
- **Pseudonymized starter slice.** `content_id` / `client_id` are pseudonyms for grouping only
  (per `flyrank-data`); this queue is scoped to Lane 2's 30k-row starter export, not the full
  warehouse, and should not be read as a client-level verdict.

In [3]:
# Receipts for the limits stated above -- not just asserted, checked
print("Client-grouped holdout performance (the honest number this playbook is built on):")
for k, v in expected_performance.items():
    print(f"  {k}: {v}")

print("\nFreshness tier sample sizes (thin-bucket caution from w04):")
print(df["freshness_tier"].value_counts())

print("\nOne row per content item -- single snapshot, no repeated per-page observations over time:")
print(len(df) == df["content_id"].nunique())

Client-grouped holdout performance (the honest number this playbook is built on):
  test_base_rate: 0.511
  precision_at_20: 0.65
  precision_at_50: 0.72
  precision_at_100: 0.66
  roc_auc: 0.583
  avg_precision: 0.577

Freshness tier sample sizes (thin-bucket caution from w04):
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64

One row per content item -- single snapshot, no repeated per-page observations over time:
True


## 3. Human review + the no-go list

**What a human must check before acting on any flagged row:**

- Read the actual page. A reason code is a hypothesis about *why* a page might be worth attention,
  not a verdict -- especially for rows in the thin `31-90`/`181+` freshness buckets, or rows whose
  CTR-laggard flag rests on a `top_3`/`striking` tier mean that a few high-volume pages could be
  skewing (the same heavy-tail warning `w04` raised about weighted vs. mean CTR).
- Check whether several top-priority rows share one `client_id`. If one client dominates the top of
  the queue, that is more likely one shared technical or template issue than several independent
  opportunities -- see the concrete count below.
- Confirm the page isn't already mid-edit, already scheduled for a rewrite, or intentionally
  deprioritized (e.g., a page the client asked to retire) before flagging it as an opportunity.
- Treat `decline_proba` as a rank, not a probability of a specific future outcome for that one
  page -- it was validated in aggregate on a held-out set of clients, not calibrated per-page.

**What must NOT be automated -- ever, in this playbook's current form:**

- **No auto-published content changes.** Title/meta rewrites and content refreshes go to a human
  editor first; the model never edits or publishes anything itself.
- **No automatic deprioritization or removal** of pages based on `decline_proba` or `not_flagged`
  status alone -- a `Stable` label just means this pass didn't flag it, not that it's safe forever.
- **No feedback loop where the model's own action queue becomes next month's training label.**
  Per the core framework's feedback-loop concern: if editors only ever touch pages this model
  flagged, the *next* label is partly shaped by this model's own picks, not by an independent
  measurement of decline -- that would quietly make the model grade its own homework.
- **No cross-client comparison or client-level performance reporting** built on this queue. The
  model was validated at the portfolio level with client identity used only for grouping the
  split, never as a client-quality signal.
- **No use in performance reviews, budget allocation, or headcount decisions** for any team or
  client. This is a content-triage tool, not a personnel or contract evaluation tool.
- **No claim that flagged pages will improve if refreshed.** Per `writing-honest-claims` and the
  `w06` audit of the paper's own refresh-multiplier finding: the honest framing is
  decision-support ("worth reviewing first, because..."), never "this fix will work."

In [4]:
# Concrete evidence for the "check client concentration" review step
top20 = queue.head(20)
client_counts = top20["client_id"].value_counts()
print("Client concentration in the top 20 priority rows:")
print(client_counts)
print(f"\nMax single-client share of the top 20: {client_counts.max()} of 20 rows")
print("-> exactly the 'one shared issue, not N independent opportunities' caution above,")
print("   made concrete instead of just asserted.")

Client concentration in the top 20 priority rows:
client_id
client_3fdba35f04    8
client_7f2253d7e2    4
client_f369cb89fc    2
client_a88a7902cb    2
client_b4944c6ff0    2
client_19581e27de    1
client_6208ef0f77    1
Name: count, dtype: int64

Max single-client share of the top 20: 8 of 20 rows
-> exactly the 'one shared issue, not N independent opportunities' caution above,
   made concrete instead of just asserted.


## 4. Monitoring / retrain triggers

What would tell a reviewer this playbook has gone stale, using the monitoring types from the
core framework (data drift, prediction drift, performance drift, segment performance):

| Signal | Reference value (this run) | Retrain / re-review trigger |
|---|---|---|
| **Data drift** -- label base rate | 51.1% (holdout), ~54% (full set) | Base rate moves more than ~5 points from this reference on a fresh export |
| **Prediction drift** -- mean `decline_proba` | ~0.50 (full set) | Mean predicted probability drifts materially without a matching base-rate shift (a sign the model, not the world, changed) |
| **Performance drift** -- precision@50 | 0.72 (client-grouped holdout) | Precision@50 on a fresh held-out batch falls to at or below the Week-4 rule baseline (~0.5, coin-flip territory per `w05`) |
| **Action-rate drift** -- flagged share | 25.0% of pages flagged (top-quartile cut by design) | Flagged share drifts far from ~25% on a fresh run without a matching change to the risk-threshold quantile |
| **Segment performance** -- per-client concentration | Up to 8/20 top-priority rows from one client (this run) | A single client repeatedly dominates the queue across multiple runs -- investigate a shared technical issue before treating it as N opportunities |

**Cadence:** re-run this scoring pass on a regular schedule (e.g. monthly, alongside a fresh data
export) rather than continuously -- the holdout evidence above is itself a snapshot, and more
frequent re-scoring buys little given how coarse (top-quartile) the risk cut already is.

In [5]:
# Reference snapshot this run establishes -- future runs compare against these numbers
reference_snapshot = {
    "holdout_base_rate": expected_performance["test_base_rate"],
    "full_set_base_rate": round(df["is_declining_label"].mean(), 3),
    "mean_decline_proba": round(df["decline_proba"].mean(), 3),
    "precision_at_50_holdout": expected_performance["precision_at_50"],
    "flagged_share": round(len(queue) / len(df), 3),
    "max_single_client_share_of_top20": int(client_counts.max()),
}
print("Reference snapshot for future drift comparison:")
for k, v in reference_snapshot.items():
    print(f"  {k}: {v}")

Reference snapshot for future drift comparison:
  holdout_base_rate: 0.511
  full_set_base_rate: 0.542
  mean_decline_proba: 0.505
  precision_at_50_holdout: 0.72
  flagged_share: 0.25
  max_single_client_share_of_top20: 8


## 5. Exports for the paper

Two kinds of file, on purpose, matching the repo's own rule (`work/README.md`): the ranked queue
is a data file and **stays out of git** (the CI leak-guard blocks dataset CSVs; this notebook
regenerates it on demand), while the metrics JSON and any figures are the **receipts** and get
committed so the capstone paper's numbers trace back to something real.

In [6]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

# 1) Ranked queue CSV -- goes to work/outputs/, gitignored by design, regenerated by this notebook
queue_out_cols = ["content_id", "client_id", "archetype", "reason_code", "action",
                   "decline_proba", "value_at_stake", "priority_score",
                   "impressions_90d", "avg_position", "ctr", "freshness_tier", "position_tier"]
queue_path = "../outputs/ranked_action_queue.csv"
queue[queue_out_cols].to_csv(queue_path, index=False)
print(f"Ranked queue written to {queue_path} ({len(queue):,} rows) -- gitignored, regenerate anytime.")

# 2) Metrics JSON -- the receipts, stays committed
metrics = {
    "model": "LogisticRegression (class_weight=balanced), same feature set as w05_model.ipynb",
    "expected_holdout_performance": expected_performance,
    "archetype_counts": df["archetype"].value_counts().to_dict(),
    "action_counts": df["action"].value_counts().to_dict(),
    "high_risk_threshold_p75": round(risk_threshold, 3),
    "reference_snapshot_for_drift_monitoring": reference_snapshot,
}
metrics_path = "../outputs/action_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics receipts written to {metrics_path} -- COMMITTED, not gitignored.")

# 3) Figure -- action queue breakdown, committed to work/figures/
action_order = ["optimize_title_meta", "refresh_content", "monitor_only", "no_action"]
counts = df["action"].value_counts().reindex(action_order)
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#4C72B0", "#DD8452", "#8C8C8C", "#CCCCCC"]
ax.barh(counts.index, counts.values, color=colors)
ax.set_xlabel("Content pages")
ax.set_title("Content action playbook -- pages by recommended action")
for i, v in enumerate(counts.values):
    ax.text(v + max(counts.values) * 0.01, i, f"{v:,}", va="center")
plt.tight_layout()
fig_path = "../figures/action_queue_by_reason_code.png"
fig.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Figure written to {fig_path} -- committed to work/figures/.")

Ranked queue written to ../outputs/ranked_action_queue.csv (7,500 rows) -- gitignored, regenerate anytime.
Metrics receipts written to ../outputs/action_playbook_metrics.json -- COMMITTED, not gitignored.
Figure written to ../figures/action_queue_by_reason_code.png -- committed to work/figures/.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.